In [1]:
import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import json
import evaluate as hf_evaluate
import os

# ==================================================================================
# CONFIGURATION
# ==================================================================================

H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_blip.json"

BATCH_SIZE = 16
EPOCHS = 30

device = torch.device("cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

# Updated from your multilabel dataset
NUM_COLORS = 9       # Black, Blue, Brown, Green, Grey, Orange, Red, White, Yellow
NUM_OBJECTS = 6      # Animal, Building, Food, Nature, Person, Vehicle
TOTAL_META_DIM = NUM_COLORS + NUM_OBJECTS  # 15

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


Using device: cpu


In [2]:
# ==================================================================================
# FIXED DATA SPLIT (NO LEAKAGE)
# ==================================================================================

def create_stratified_split(total_samples, group_size=5):
    """
    Split data ensuring groups of 5 consecutive samples stay together.
    For each group: 3 train, 1 val, 1 test
    """
    num_groups = total_samples // group_size
    train_indices = []
    val_indices = []
    test_indices = []
    
    for group_idx in range(num_groups):
        start_idx = group_idx * group_size
        group_indices = list(range(start_idx, start_idx + group_size))
        
        # Deterministic split within each group
        train_indices.extend(group_indices[:3])  # First 3 for training
        val_indices.append(group_indices[3])     # 4th for validation
        test_indices.append(group_indices[4])    # 5th for testing
    
    # Handle remainder if total_samples not divisible by 5
    remainder = total_samples % group_size
    if remainder > 0:
        start_idx = num_groups * group_size
        remainder_indices = list(range(start_idx, total_samples))
        
        if remainder >= 3:
            train_indices.extend(remainder_indices[:3])
            if remainder >= 4:
                val_indices.append(remainder_indices[3])
            if remainder == 5:
                test_indices.append(remainder_indices[4])
        else:
            train_indices.extend(remainder_indices)
    
    print(f"Split Statistics:")
    print(f"  Total Groups: {num_groups}")
    print(f"  Train: {len(train_indices)} samples ({len(train_indices)/total_samples*100:.1f}%)")
    print(f"  Val:   {len(val_indices)} samples ({len(val_indices)/total_samples*100:.1f}%)")
    print(f"  Test:  {len(test_indices)} samples ({len(test_indices)/total_samples*100:.1f}%)")
    
    return train_indices, val_indices, test_indices

In [3]:
# ==================================================================================
# GRANGER CAUSALITY
# ==================================================================================
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[1].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j:
                continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            min_len = 20
            if len(ts_i) < min_len or len(ts_j) < min_len:
                causality_matrix[i, j] = 0.0
                continue
            data = np.vstack([ts_j, ts_i]).T
            try:
                current_maxlag = min(5, len(data)//2 - 2)
                if current_maxlag < 1:
                    causality_matrix[i, j] = 0.0
                    continue
                results = grangercausalitytests(data, maxlag=current_maxlag, verbose=False)
                p_value = results[current_maxlag][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except Exception as e:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)

    if edge_attr is None:
        edge_attr = torch.tensor([], dtype=torch.float)
    elif edge_attr.ndim == 0:
        edge_attr = edge_attr.unsqueeze(0)

    return edge_index.to(torch.long), edge_attr.to(torch.float)

In [4]:
# ==================================================================================
# DATASET
# ==================================================================================
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')

        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))

        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

In [5]:
# ==================================================================================
# MODEL COMPONENTS
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]

        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))

        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        return encoder_outputs, encoder_hidden


class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        attn_energies = self.attn(encoder_outputs)
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)


class MetadataEncoder(nn.Module):
    """Multi-label color and object encoder"""
    def __init__(self, num_colors, num_objects, 
                 color_feature_dim=32, object_feature_dim=32):
        super().__init__()
        
        self.color_processor = nn.Sequential(
            nn.Linear(num_colors, 64),
            nn.ReLU(),
            nn.Linear(64, color_feature_dim)
        )
        
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 64),
            nn.ReLU(),
            nn.Linear(64, object_feature_dim)
        )

        self.output_dim = color_feature_dim + object_feature_dim

    def forward(self, metadata):
        color_input = metadata[:, :NUM_COLORS].float()
        object_input = metadata[:, NUM_COLORS:].float()

        color_vec = self.color_processor(color_input)
        object_vec = self.object_processor(object_input)

        combined_features = torch.cat([color_vec, object_vec], dim=1)
        return combined_features


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)

        self.rnn_input_dim = emb_dim + enc_dim + meta_features_dim + enc_dim
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)

        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        decoder_initial_hidden = bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)
        return decoder_initial_hidden

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)
        context_permuted = context.permute(1, 0, 2)

        rnn_input = torch.cat((
            embedded,
            context_permuted,
            meta_features_unsqueezed,
            global_eeg_context_unsqueezed
        ), dim=2)

        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))

        return prediction, hidden, context.squeeze(1)

In [6]:
# ==================================================================================
# CURRICULUM MULTI-TASK LOSS (FIXED VERSION)
# ==================================================================================
class CurriculumMultiTaskLoss(nn.Module):
    """
    Curriculum learning with DECAYING auxiliary task weights.
    Text task maintains high weight, auxiliary tasks decay over time.
    """
    def __init__(self, initial_aux_weight=2.0, min_aux_weight=0.1, decay_epochs=20):
        super().__init__()
        self.initial_aux_weight = initial_aux_weight
        self.min_aux_weight = min_aux_weight
        self.decay_epochs = decay_epochs
        self.current_epoch = 0
    
    def get_aux_weight(self):
        """Exponential decay of auxiliary task weight"""
        if self.current_epoch >= self.decay_epochs:
            return self.min_aux_weight
        
        # Exponential decay from initial to min
        progress = self.current_epoch / self.decay_epochs
        weight = self.initial_aux_weight * (self.min_aux_weight / self.initial_aux_weight) ** progress
        return weight
    
    def forward(self, loss_t, loss_c, loss_o):
        aux_weight = self.get_aux_weight()
        
        # Text gets weight of 1.0 (prioritized)
        # Auxiliary tasks get decaying weight
        total_loss = loss_t + aux_weight * (loss_c + loss_o)
        
        return total_loss, aux_weight
    
    def step_epoch(self):
        """Call this at the end of each epoch"""
        self.current_epoch += 1

In [7]:
# ==================================================================================
# DIVERSITY LOSS
# ==================================================================================
class DiversityLoss(nn.Module):
    """Encourages diverse token predictions"""
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
    
    def forward(self, logits):
        batch_size, seq_len, vocab_size = logits.shape
        probs = F.softmax(logits, dim=-1)
        avg_probs = probs.mean(dim=(0, 1))
        uniform = torch.ones_like(avg_probs) / self.vocab_size
        kl_div = F.kl_div(avg_probs.log(), uniform, reduction='batchmean')
        return kl_div

In [8]:
# ==================================================================================
# CALIBRATED MULTI-LABEL PREDICTION
# ==================================================================================
def calibrated_multilabel_predict(logits, threshold=0.6, max_labels=4):
    """
    Conservative multi-label prediction with:
    1. Higher threshold (0.6 vs 0.5)
    2. Maximum label limit
    3. Top-k selection if over limit
    """
    probs = torch.sigmoid(logits)
    
    # Apply threshold
    predictions = (probs > threshold).float()
    
    # Limit maximum labels
    num_predicted = predictions.sum(dim=1)
    for i in range(predictions.size(0)):
        if num_predicted[i] > max_labels:
            # Keep only top-k predictions
            top_k_values, top_k_indices = torch.topk(probs[i], max_labels)
            predictions[i] = 0.0
            predictions[i][top_k_indices] = 1.0
    
    return predictions

In [9]:
# ==================================================================================
# MAIN MODEL
# ==================================================================================
class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        self.meta_encoder = MetadataEncoder(num_colors, num_objects)

        meta_features_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2

        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                               meta_features_dim, dec_layers, pad_id, dropout)

        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_objects)
        )
        self.num_colors = num_colors
        self.num_objects = num_objects

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, teacher_forcing_ratio=0.5):
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        target_vocab_size = self.decoder.vocab_size

        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        meta_features = self.meta_encoder(metadata)

        decoder_hidden = self.decoder.init_hidden(encoder_hidden)

        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, batch_size, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

        meta_preds = self.meta_head(global_eeg_context)
        pred_color_logits = meta_preds[:, :self.num_colors]
        pred_object_logits = meta_preds[:, self.num_colors:]

        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]

        for t in range(1, target_len):
            output, decoder_hidden, _ = self.decoder(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                meta_features,
                global_eeg_context
            )

            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1

        return outputs[1:].permute(1, 0, 2), pred_color_logits, pred_object_logits

In [10]:
# ==================================================================================
# TRAINING AND EVALUATION
# ==================================================================================

def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion, object_criterion,
                   diversity_criterion, curriculum_loss, granger_edge_index, granger_edge_attr,
                   diversity_weight=0.01):
    model.train()
    total_loss = 0.0
    total_loss_components = {'text': 0.0, 'color': 0.0, 'object': 0.0, 'diversity': 0.0}
    
    progress_bar = tqdm(loader, desc="Training", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        optimizer.zero_grad()

        text_logits, pred_color_logits, pred_object_logits = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.5
        )

        # Individual losses
        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color_logits, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object_logits, meta_b[:, NUM_COLORS:].float())
        loss_div = diversity_criterion(text_logits)

        # Curriculum-weighted combination (auxiliary tasks decay over time)
        loss_main, aux_weight = curriculum_loss(loss_t, loss_c, loss_o)
        
        # Total loss
        loss = loss_main + diversity_weight * loss_div

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_loss_components['text'] += loss_t.item()
        total_loss_components['color'] += loss_c.item()
        total_loss_components['object'] += loss_o.item()
        total_loss_components['diversity'] += loss_div.item()

        progress_bar.set_postfix(
            loss=loss.item(), 
            txt=loss_t.item(),
            aux_w=aux_weight
        )

    n = len(loader)
    return {k: v/n for k, v in total_loss_components.items()}, total_loss / n, aux_weight


@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, object_criterion,
            diversity_criterion, curriculum_loss, granger_edge_index, granger_edge_attr,
            diversity_weight=0.01):
    model.eval()
    total_loss = 0.0
    total_loss_components = {'text': 0.0, 'color': 0.0, 'object': 0.0, 'diversity': 0.0}
    
    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        text_logits, pred_color_logits, pred_object_logits = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.0
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color_logits, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object_logits, meta_b[:, NUM_COLORS:].float())
        loss_div = diversity_criterion(text_logits)

        loss_main, aux_weight = curriculum_loss(loss_t, loss_c, loss_o)
        loss = loss_main + diversity_weight * loss_div

        total_loss += loss.item()
        total_loss_components['text'] += loss_t.item()
        total_loss_components['color'] += loss_c.item()
        total_loss_components['object'] += loss_o.item()
        total_loss_components['diversity'] += loss_div.item()

        progress_bar.set_postfix(loss=loss.item())

    n = len(loader)
    return {k: v/n for k, v in total_loss_components.items()}, total_loss / n

In [11]:
# ==================================================================================
# INFERENCE
# ==================================================================================

@torch.no_grad()
def generate_with_context_boost(model, eeg_signal, meta_signal, edge_index, edge_attr,
                                k=5, penalty_alpha=0.3, context_beta=0.7, max_len=100):
    model.eval()
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    meta_signal = meta_signal.unsqueeze(0).to(device)

    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
    meta_features = model.meta_encoder(meta_signal)

    decoder_hidden = model.decoder.init_hidden(encoder_hidden)

    hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
    last_layer_hidden = hidden_reshaped[-1]
    global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

    meta_preds = model.meta_head(global_eeg_context)
    pred_color_logits = meta_preds[0, :model.num_colors]
    pred_object_logits = meta_preds[0, model.num_objects:]

    # Use calibrated prediction
    pred_colors = calibrated_multilabel_predict(pred_color_logits.unsqueeze(0), threshold=0.6, max_labels=3)
    pred_objects = calibrated_multilabel_predict(pred_object_logits.unsqueeze(0), threshold=0.6, max_labels=3)
    
    pred_color_ids = pred_colors[0].nonzero(as_tuple=True)[0].tolist()
    pred_object_ids = pred_objects[0].nonzero(as_tuple=True)[0].tolist()

    generated_ids = torch.tensor([SOS_ID], device=device)

    for step in range(max_len):
        input_token = generated_ids[-1].unsqueeze(0)

        prediction, new_hidden, attention_context = model.decoder(
            input_token,
            decoder_hidden,
            encoder_outputs,
            meta_features,
            global_eeg_context
        )

        decoder_hidden = new_hidden
        model_log_probs = F.log_softmax(prediction, dim=-1).squeeze(0)
        
        current_seq_len = generated_ids.shape[0]
        prev_token_embeddings = F.normalize(model.decoder.embedding(generated_ids), dim=-1)
        
        topk_model_log_probs, topk_ids = torch.topk(model_log_probs, k)
        candidate_token_embeddings = F.normalize(model.decoder.embedding(topk_ids), dim=-1)
        
        sim_matrix = torch.matmul(candidate_token_embeddings, prev_token_embeddings.t())
        degeneration_penalty = torch.zeros(k, device=device)
        if current_seq_len > 1:
            degeneration_penalty, _ = torch.max(sim_matrix, dim=-1)
            
        current_decoder_state = F.normalize(decoder_hidden[-1].squeeze(), dim=-1)
        context_agreement_score = torch.matmul(candidate_token_embeddings, current_decoder_state)
        
        final_score = topk_model_log_probs + context_beta * context_agreement_score - penalty_alpha * degeneration_penalty
        
        best_next_token_idx = torch.argmax(final_score)
        next_token_id = topk_ids[best_next_token_idx]

        generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(0)])
        if next_token_id.item() == EOS_ID:
            break
            
    if generated_ids.numel() > 1:
        predicted_text_ids = generated_ids[1:-1] if generated_ids[-1].item() == EOS_ID else generated_ids[1:]
        predicted_text = tokenizer.decode(predicted_text_ids.tolist(), skip_special_tokens=True)
    else:
        predicted_text = ""
    
    return predicted_text, pred_color_ids, pred_object_ids

In [ ]:
'''# ==================================================================================
# MAIN EXECUTION
# ==================================================================================

if __name__ == "__main__":
    # Create dataset with FIXED split
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    N = len(dataset)
    
    print(f"\n=== Creating Stratified Data Split (No Leakage) ===")
    train_indices, val_indices, test_indices = create_stratified_split(N, group_size=5)
    
    train_ds = Subset(dataset, train_indices)
    val_ds = Subset(dataset, val_indices)
    test_ds = Subset(dataset, test_indices)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

    # Create Granger matrix
    print("\n=== Creating Granger Causality Matrix ===")
    try:
        eeg_b, _, _ = next(iter(train_loader))
        granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
        num_channels = eeg_b.shape[1]
        granger_edge_index, granger_edge_attr = add_self_loops(
            granger_edge_index, edge_attr=granger_edge_attr, num_nodes=num_channels, fill_value=1.0
        )
        if granger_edge_attr is None:
            granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float)
        granger_edge_index = granger_edge_index.to(torch.long).to(device)
        granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
        print(f"Granger matrix created: {granger_edge_index.shape}")
    except Exception as e:
        print(f"Error: {e}. Using fallback graph.")
        num_channels = 62
        edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
        granger_edge_index = edge_index.to(torch.long).to(device)
        granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float32).to(device)

    # Instantiate model
    print("\n=== Initializing Model ===")
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS,
        num_objects=NUM_OBJECTS,
        pad_id=PAD_ID,
        dropout=0.2,
        enc_hidden=256,
        dec_hidden=256,
        emb_dim=256,
        dec_layers=2
    ).to(device)

    print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # Loss functions (with pos_weight from manifest)
    object_pos_weight = torch.tensor([3.1176, 5.6667, 6.1066, 1.1021, 2.0905, 7.5366]).to(device)
    color_pos_weight = torch.tensor([3.1543, 1.2764, 3.9645, 1.7888, 0.8301, 11.2807, 5.2780, 1.0408, 4.4054]).to(device)

    text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    color_criterion = nn.BCEWithLogitsLoss(pos_weight=color_pos_weight)
    object_criterion = nn.BCEWithLogitsLoss(pos_weight=object_pos_weight)
    diversity_criterion = DiversityLoss(TEXT_VOCAB_SIZE).to(device)
    
    # FIXED: Curriculum loss with DECAYING auxiliary weights
    curriculum_loss = CurriculumMultiTaskLoss(
        initial_aux_weight=2.0,  # Start with aux tasks at 2x weight
        min_aux_weight=0.1,      # Decay to 0.1x by end
        decay_epochs=20
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=3, verbose=True)

    DIVERSITY_WEIGHT = 0.01
    best_val_loss = float('inf')

    print("\n=== Starting Training with Curriculum MTL ===")
    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()

        train_components, train_loss, aux_weight = train_one_epoch(
            model, train_loader, optimizer,
            text_criterion, color_criterion, object_criterion,
            diversity_criterion, curriculum_loss,
            granger_edge_index, granger_edge_attr, DIVERSITY_WEIGHT
        )
        
        val_components, val_loss = evaluate(
            model, val_loader,
            text_criterion, color_criterion, object_criterion,
            diversity_criterion, curriculum_loss,
            granger_edge_index, granger_edge_attr, DIVERSITY_WEIGHT
        )

        curriculum_loss.step_epoch()  # Decay auxiliary weight
        scheduler.step(val_loss)
        end_time = time.time()
        
        print(f'\nEpoch: {epoch:02} | Time: {int(end_time - start_time)}s')
        print(f'  Train Loss: {train_loss:.4f} | Text: {train_components["text"]:.4f} | Div: {train_components["diversity"]:.4f}')
        print(f'  Val Loss:   {val_loss:.4f} | Text: {val_components["text"]:.4f} | Div: {val_components["diversity"]:.4f}')
        print(f'  Aux Weight: {aux_weight:.3f} (Color: {train_components["color"]:.4f}, Obj: {train_components["object"]:.4f})')

        # Diversity check every 5 epochs
        if epoch % 5 == 0:
            print("  Checking diversity...")
            sample_outputs = []
            for j in range(10):
                pred, _, _ = generate_with_context_boost(
                    model, test_ds[j][0], test_ds[j][1],
                    granger_edge_index, granger_edge_attr
                )
                sample_outputs.append(pred)
            unique_rate = len(set(sample_outputs)) / len(sample_outputs)
            print(f"  Diversity: {unique_rate*100:.0f}% unique")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'eeg-text-curriculum-fixed.pt')
            print("  ✓ Model saved")

    print("\n=== Training Complete ===\n")

    # INFERENCE
    model.load_state_dict(torch.load('eeg-text-curriculum-fixed.pt', map_location=device))
    print("Best model loaded for inference.\n")

    # Load object mapping
    try:
        with open(OBJECT_MAPPING_FILE, 'r') as f:
            object_mapping = json.load(f)
    except:
        object_mapping = {}

    color_mapping = {0: "Black", 1: "Blue", 2: "Brown", 3: "Green", 4: "Grey", 
                     5: "Orange", 6: "Red", 7: "White", 8: "Yellow"}

    print("=== Running Inference ===")
    predictions = []
    references = []
    
    NUM_SAMPLES = 20
    for i in range(NUM_SAMPLES):
        eeg_sample, meta_sample, true_text_ids = test_ds[i]
        
        predicted_text, pred_color_ids, pred_object_ids = generate_with_context_boost(
            model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr
        )

        true_text = tokenizer.decode(true_text_ids.tolist(), skip_special_tokens=True)
        predictions.append(predicted_text)
        references.append(true_text)

        # Get ground truth
        true_color_ids = meta_sample[:NUM_COLORS].nonzero(as_tuple=True)[0].tolist()
        true_object_ids = meta_sample[NUM_COLORS:].nonzero(as_tuple=True)[0].tolist()

        print(f"\n--- Sample {i+1}/{NUM_SAMPLES} ---")
        print(f"GT:   {true_text}")
        print(f"Pred: {predicted_text}")
        print(f"Colors - GT: {len(true_color_ids)}, Pred: {len(pred_color_ids)}")
        print(f"Objects - GT: {len(true_object_ids)}, Pred: {len(pred_object_ids)}")

    # Metrics
    try:
        bleu_metric = hf_evaluate.load('bleu')
        bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
        
        rouge_metric = hf_evaluate.load('rouge')
        rouge_results = rouge_metric.compute(predictions=predictions, references=references)
        
        print(f"\n{'='*50}")
        print(f"FINAL RESULTS (No Data Leakage)")
        print(f"{'='*50}")
        print(f"BLEU:    {bleu_results['bleu']:.4f}")
        print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
        print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")
        print(f"{'='*50}")
    except Exception as e:
        print(f"Error computing metrics: {e}")'''

In [12]:
# ==========================================
# BLOCK 1 — DATA, DATALOADERS, GRANGER GRAPH
# ==========================================

# Create dataset with FIXED split
dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
N = len(dataset)

print("\n=== Creating Stratified Data Split (No Leakage) ===")
train_indices, val_indices, test_indices = create_stratified_split(N, group_size=5)

train_ds = Subset(dataset, train_indices)
val_ds = Subset(dataset, val_indices)
test_ds = Subset(dataset, test_indices)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_multimodal_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_multimodal_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_multimodal_batch)

# Create Granger matrix
print("\n=== Creating Granger Causality Matrix ===")
try:
    eeg_b, _, _ = next(iter(train_loader))
    granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)

    num_channels = eeg_b.shape[1]
    granger_edge_index, granger_edge_attr = add_self_loops(
        granger_edge_index, edge_attr=granger_edge_attr,
        num_nodes=num_channels, fill_value=1.0
    )

    if granger_edge_attr is None:
        granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float)

    granger_edge_index = granger_edge_index.to(torch.long).to(device)
    granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)

    print(f"Granger matrix created: {granger_edge_index.shape}")

except Exception as e:
    print(f"Error: {e}. Using fallback graph.")
    num_channels = 62

    edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)

    granger_edge_index = edge_index.to(torch.long).to(device)
    granger_edge_attr = torch.ones(granger_edge_index.shape[1],
                                   dtype=torch.float32).to(device)



=== Creating Stratified Data Split (No Leakage) ===
Split Statistics:
  Total Groups: 5600
  Train: 16800 samples (60.0%)
  Val:   5600 samples (20.0%)
  Test:  5600 samples (20.0%)

=== Creating Granger Causality Matrix ===


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Granger matrix created: torch.Size([2, 1942])


In [ ]:
# ==========================================
# BLOCK 2 — MODEL + TRAINING LOOP
# ==========================================

print("\n=== Initializing Model ===")
model = Seq2Seq(
    text_vocab_size=TEXT_VOCAB_SIZE,
    num_colors=NUM_COLORS,
    num_objects=NUM_OBJECTS,
    pad_id=PAD_ID,
    dropout=0.2,
    enc_hidden=256,
    dec_hidden=256,
    emb_dim=256,
    dec_layers=2
).to(device)

print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Loss functions
object_pos_weight = torch.tensor([3.1176, 5.6667, 6.1066, 1.1021, 2.0905, 7.5366]).to(device)
color_pos_weight = torch.tensor([3.1543, 1.2764, 3.9645, 1.7888, 0.8301,
                                 11.2807, 5.2780, 1.0408, 4.4054]).to(device)

text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
color_criterion = nn.BCEWithLogitsLoss(pos_weight=color_pos_weight)
object_criterion = nn.BCEWithLogitsLoss(pos_weight=object_pos_weight)
diversity_criterion = DiversityLoss(TEXT_VOCAB_SIZE).to(device)

curriculum_loss = CurriculumMultiTaskLoss(
    initial_aux_weight=2.0,
    min_aux_weight=0.1,
    decay_epochs=20
).to(device)

optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=3, verbose=True)

DIVERSITY_WEIGHT = 0.01
best_val_loss = float('inf')

print("\n=== Starting Training with Curriculum MTL ===")
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_components, train_loss, aux_weight = train_one_epoch(
        model, train_loader, optimizer,
        text_criterion, color_criterion, object_criterion,
        diversity_criterion, curriculum_loss,
        granger_edge_index, granger_edge_attr, DIVERSITY_WEIGHT
    )
    
    val_components, val_loss = evaluate(
        model, val_loader,
        text_criterion, color_criterion, object_criterion,
        diversity_criterion, curriculum_loss,
        granger_edge_index, granger_edge_attr, DIVERSITY_WEIGHT
    )

    curriculum_loss.step_epoch()
    scheduler.step(val_loss)
    end_time = time.time()
    
    print(f'\nEpoch: {epoch:02} | Time: {int(end_time - start_time)}s')
    print(f'  Train Loss: {train_loss:.4f} | Text: {train_components["text"]:.4f} | Div: {train_components["diversity"]:.4f}')
    print(f'  Val Loss:   {val_loss:.4f} | Text: {val_components["text"]:.4f} | Div: {val_components["diversity"]:.4f}')
    print(f'  Aux Weight: {aux_weight:.3f}')

    if epoch % 5 == 0:
        print("  Checking diversity...")
        sample_outputs = []
        for j in range(10):
            pred, _, _ = generate_with_context_boost(
                model, test_ds[j][0], test_ds[j][1],
                granger_edge_index, granger_edge_attr
            )
            sample_outputs.append(pred)
        unique_rate = len(set(sample_outputs)) / len(sample_outputs)
        print(f"  Diversity: {unique_rate*100:.0f}% unique")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'eeg-text-curriculum-fixed.pt')
        print("  ✓ Model saved")

print("\n=== Training Complete ===\n")



=== Initializing Model ===
Total parameters: 19,740,617

=== Starting Training with Curriculum MTL ===


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training:   0%|          | 0/1050 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/350 [00:00<?, ?it/s]


Epoch: 01 | Time: 380s
  Train Loss: 9.5552 | Text: 5.3915 | Div: 0.0001
  Val Loss:   8.5462 | Text: 4.4971 | Div: 0.0001
  Aux Weight: 2.000
  ✓ Model saved


...


Epoch: 03 | Time: 367s
  Train Loss: 6.7765 | Text: 3.7494 | Div: 0.0001
  Val Loss:   7.1882 | Text: 4.1912 | Div: 0.0001
  Aux Weight: 1.482
  ✓ Model saved


Training:   0%|          | 0/1050 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/350 [00:00<?, ?it/s]


Epoch: 04 | Time: 366s
  Train Loss: 6.1024 | Text: 3.5068 | Div: 0.0001
  Val Loss:   6.7616 | Text: 4.1790 | Div: 0.0001
  Aux Weight: 1.276
  ✓ Model saved


Training:   0%|          | 0/1050 [00:00<?, ?it/s]

In [18]:
# ==========================================
# BLOCK 3 — INFERENCE + METRICS
# ==========================================
model = Seq2Seq(
    text_vocab_size=TEXT_VOCAB_SIZE,
    num_colors=NUM_COLORS,
    num_objects=NUM_OBJECTS,
    pad_id=PAD_ID,
    dropout=0.2,
    enc_hidden=256,
    dec_hidden=256,
    emb_dim=256,
    dec_layers=2
).to(device)

model.load_state_dict(torch.load('eeg-text-curriculum-fixed.pt', map_location=device))
print("Best model loaded for inference.\n")

try:
    with open(OBJECT_MAPPING_FILE, 'r') as f:
        object_mapping = json.load(f)
except:
    object_mapping = {}

color_mapping = {
    0: "Black", 1: "Blue", 2: "Brown", 3: "Green", 4: "Grey",
    5: "Orange", 6: "Red", 7: "White", 8: "Yellow"
}

print("=== Running Inference ===")
predictions = []
references = []

NUM_SAMPLES = 40
for i in range(NUM_SAMPLES):
    eeg_sample, meta_sample, true_text_ids = test_ds[i]
    
    predicted_text, pred_color_ids, pred_object_ids = generate_with_context_boost(
        model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr
    )

    true_text = tokenizer.decode(true_text_ids.tolist(), skip_special_tokens=True)
    predictions.append(predicted_text)
    references.append(true_text)

    true_color_ids = meta_sample[:NUM_COLORS].nonzero(as_tuple=True)[0].tolist()
    true_object_ids = meta_sample[NUM_COLORS:].nonzero(as_tuple=True)[0].tolist()

    print(f"\n--- Sample {i+1}/{NUM_SAMPLES} ---")
    print(f"GT:   {true_text}")
    print(f"Pred: {predicted_text}")
    print(f"Colors - GT: {len(true_color_ids)}, Pred: {len(pred_color_ids)}")
    print(f"Objects - GT: {len(true_object_ids)}, Pred: {len(pred_object_ids)}")

# Compute Metrics
try:
    bleu_metric = hf_evaluate.load('bleu')
    bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    
    rouge_metric = hf_evaluate.load('rouge')
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    
    print("\n" + "="*50)
    print("FINAL RESULTS (No Data Leakage)")
    print("="*50)
    print(f"BLEU:    {bleu_results['bleu']:.4f}")
    print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
    print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")
    print("="*50)

except Exception as e:
    print(f"Error computing metrics: {e}")


Best model loaded for inference.

=== Running Inference ===

--- Sample 1/40 ---
GT:   a city at night with buildings lit up
Pred: a large with a buildings and a
Colors - GT: 3, Pred: 0
Objects - GT: 2, Pred: 0

--- Sample 2/40 ---
GT:   a beach with a cloudy sky and a beach
Pred: a large with a buildings and a
Colors - GT: 3, Pred: 0
Objects - GT: 1, Pred: 0

--- Sample 3/40 ---
GT:   a blue jellyfish in the dark
Pred: a person is cutting a water
Colors - GT: 3, Pred: 0
Objects - GT: 0, Pred: 0

--- Sample 4/40 ---
GT:   four small rabbits in a bowl
Pred: a panda bear is sitting on the ground
Colors - GT: 3, Pred: 0
Objects - GT: 1, Pred: 0

--- Sample 5/40 ---
GT:   a man is snowboarding on a snowy mountain
Pred: a man in a black shirt is dancing
Colors - GT: 3, Pred: 0
Objects - GT: 1, Pred: 0

--- Sample 6/40 ---
GT:   a forest with trees and bushes in the fore
Pred: a large with a and trees and it
Colors - GT: 3, Pred: 0
Objects - GT: 1, Pred: 0

--- Sample 7/40 ---
GT:   a panda 